<a href="https://colab.research.google.com/github/G0nkly/pytorch_sandbox/blob/main/gpts/microGPT/microgpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import math
import random
random.seed(42)

In [ ]:
if not os.path.exists("input.txt"):
  import urllib.request
  names_url = "https://raw.githubusercontent.com/karpathy/makemore/988aa59/names.txt"
  urllib.request.urlretrieve(names_url, "input.txt")

docs = [line.strip() for line in open("input.txt") if line.strip()]
random.shuffle(docs)
print(f"num docs: {len(docs)}")

num docs: 32033


In [ ]:
uchars = sorted(set("".join(docs)))
BOS = len(uchars) # Beginning of Sequence
vocab_size = len(uchars) + 1
print(f"vocab size: {vocab_size}")

vocab size: 27


In [ ]:
class Value:
  __slots__ = ("data", "grad", "_children", "_local_grads")

  def __init__(self, data, children=(), _local_grads=()):
    self.data = data                  # scalar value of this node calculated during forward pass
    self.grad = 0                     # derivative of the loss w.r.t this node, calculated in backward pass
    self._children = children         # children of this node in the computation graph
    self._local_grads = _local_grads  # local derivative of this node w.r.t its childre

  def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    return Value(self.data + other.data, (self, other), (1, 1))

  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    return Value(self.data * other.data, (self, other), (other.data, self.data))

  def __pow__(self, other):
    return Value(self.data**other, (self,), (other * self.data**(other-1),)

  def log(self):
    return Value(math.log(self.data), (self,), (1/self.data,))

  def exp(self):
    return Value(math.exp(self.data), (self,), (math.exp(self.data),))